In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import cv2

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import tensorflow as tf
from tensorflow.keras import layers, models

2026-05-19 15:27:11.786007: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779204431.981577      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779204432.041409      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779204432.499164      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779204432.499218      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779204432.499221      16 computation_placer.cc:177] computation placer alr

In [2]:
# -----------------------------
# CLEAN DATASET
# -----------------------------

kodak_clean_dir = Path(
    "/kaggle/input/datasets/sherylmehta/kodak-dataset"
)

bsd500_clean_dir = Path(
    "/kaggle/input/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500/images"
)

div2k_clean_base_dir = Path(
    "/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images"
)


# -----------------------------
# BSD500 CLEAN SPLIT
# -----------------------------

bsd500_train_clean_dir = bsd500_clean_dir / "train"
bsd500_val_clean_dir   = bsd500_clean_dir / "val"
bsd500_test_clean_dir  = bsd500_clean_dir / "test"


# -----------------------------
# DIV2K CLEAN SPLIT
# -----------------------------

div2k_train_clean_dir = div2k_clean_base_dir / "DIV2K_train_HR" / "DIV2K_train_HR"
div2k_valid_clean_dir = div2k_clean_base_dir / "DIV2K_valid_HR" / "DIV2K_valid_HR"

In [3]:
# -----------------------------
# DIV2K CLEAN SPLIT
# -----------------------------

div2k_train_clean_dir = div2k_clean_base_dir / "DIV2K_train_HR" / "DIV2K_train_HR"
div2k_valid_clean_dir = div2k_clean_base_dir / "DIV2K_valid_HR" / "DIV2K_valid_HR"


# -----------------------------
# OUTPUT NOISY DATASET FOR N2N - POISSON
# -----------------------------
# Per Noise2Noise servono due versioni rumorose indipendenti:
# noisy1 = input della rete
# noisy2 = target della rete
#


n2n_poisson_base_dir = Path("/kaggle/working/n2n_poisson_datasets")


kodak_noisy1_dir = n2n_poisson_base_dir / "kodak_lambda10" / "noisy1"
kodak_noisy2_dir = n2n_poisson_base_dir / "kodak_lambda10" / "noisy2"

# BSD500

bsd500_noisy1_dir = n2n_poisson_base_dir / "bsd500_lambda10" / "noisy1"
bsd500_noisy2_dir = n2n_poisson_base_dir / "bsd500_lambda10" / "noisy2"

# DIV2K

div2k_noisy1_base_dir = n2n_poisson_base_dir / "div2k_lambda10" / "noisy1"
div2k_noisy2_base_dir = n2n_poisson_base_dir / "div2k_lambda10" / "noisy2"


# -----------------------------
# OUTPUT MODEL / DENOISED - POISSON
# -----------------------------

n2n_output_dir = Path("/kaggle/working/n2n_poisson_outputs")
n2n_output_dir.mkdir(parents=True, exist_ok=True)

In [4]:
def get_image_paths(input_dir):
    """
    Restituisce tutti i path delle immagini contenute in input_dir,
    includendo eventuali sottocartelle.
    """

    input_dir = Path(input_dir)

    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]

    image_paths = [
        p for p in input_dir.rglob("*")
        if p.suffix.lower() in image_extensions
    ]

    return sorted(image_paths)

In [5]:
def add_poisson_noise_opencv(image, lambda_value=10, seed=None):
    """
    Aggiunge rumore Poisson a un'immagine uint8 in [0, 255].

    image: immagine uint8 in [0, 255]
    lambda_value: parametro che controlla l'intensità del rumore Poisson
    seed: seed opzionale per rendere la generazione riproducibile

    L'immagine viene normalizzata in [0, 1], poi viene applicato un modello
    Poisson tramite:

        noisy = Poisson(image_norm * lambda_value) / lambda_value

    In questo modo lambda_value controlla il livello di quantizzazione/rumore:
    valori più piccoli producono rumore più forte, valori più grandi rumore più debole.

    Ritorna:
        noisy_image: immagine uint8 in [0, 255]
    """

    rng = np.random.default_rng(seed)

    # Conversione immagine in float [0, 1]
    image_float = image.astype(np.float32) / 255.0

    # Applica rumore Poisson
    noisy_float = rng.poisson(image_float * lambda_value) / lambda_value

    # Riporta i valori nel range valido [0, 1]
    noisy_float = np.clip(noisy_float, 0, 1)

    # Conversione finale in uint8 [0, 255]
    noisy_image = (noisy_float * 255).astype(np.uint8)

    return noisy_image

In [6]:
def create_poisson_n2n_pair_dataset(
    input_dir,
    output_dir_noisy1,
    output_dir_noisy2,
    lambda_value=10,
    seed1=42,
    seed2=123
):
    """
    Crea due versioni rumorose indipendenti dello stesso dataset clean
    usando rumore Poisson.

    Queste due versioni servono per Noise2Noise:

        input  = noisy1
        target = noisy2

    Entrambe hanno lo stesso parametro lambda_value.
    Cambia solo la realizzazione casuale del rumore, controllata da seed diversi.

    Per Noise2Noise NON bisogna cambiare lambda tra noisy1 e noisy2:
    bisogna mantenere lo stesso livello di rumore e cambiare solo il seed.
    """

    input_dir = Path(input_dir)
    output_dir_noisy1 = Path(output_dir_noisy1)
    output_dir_noisy2 = Path(output_dir_noisy2)

    output_dir_noisy1.mkdir(parents=True, exist_ok=True)
    output_dir_noisy2.mkdir(parents=True, exist_ok=True)

    image_paths = get_image_paths(input_dir)

    print("Cartella input clean:", input_dir)
    print("Output noisy1:", output_dir_noisy1)
    print("Output noisy2:", output_dir_noisy2)
    print("Lambda Poisson:", lambda_value)
    print("Immagini trovate:", len(image_paths))

    psnr_values_noisy1 = []
    psnr_values_noisy2 = []
    processed_images = []

    for idx, img_path in enumerate(tqdm(image_paths)):

        image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)

        if image is None:
            print("Immagine non letta:", img_path)
            continue

        noisy1 = add_poisson_noise_opencv(
            image,
            lambda_value=lambda_value,
            seed=seed1 + idx
        )

        noisy2 = add_poisson_noise_opencv(
            image,
            lambda_value=lambda_value,
            seed=seed2 + idx
        )

        relative_path = img_path.relative_to(input_dir)

        save_path_noisy1 = output_dir_noisy1 / relative_path
        save_path_noisy2 = output_dir_noisy2 / relative_path

        save_path_noisy1.parent.mkdir(parents=True, exist_ok=True)
        save_path_noisy2.parent.mkdir(parents=True, exist_ok=True)

        success1 = cv2.imwrite(str(save_path_noisy1), noisy1)
        success2 = cv2.imwrite(str(save_path_noisy2), noisy2)

        if not success1:
            print("Errore nel salvataggio noisy1:", save_path_noisy1)
            continue

        if not success2:
            print("Errore nel salvataggio noisy2:", save_path_noisy2)
            continue

        psnr_values_noisy1.append(
            psnr(image, noisy1, data_range=255)
        )

        psnr_values_noisy2.append(
            psnr(image, noisy2, data_range=255)
        )

        processed_images.append(img_path.name)

    print("Immagini processate:", len(processed_images))

    if len(psnr_values_noisy1) > 0:
        print("PSNR medio clean/noisy1:", np.mean(psnr_values_noisy1))
        print("PSNR medio clean/noisy2:", np.mean(psnr_values_noisy2))
        
    return image_paths, processed_images, psnr_values_noisy1, psnr_values_noisy2

In [7]:
create_poisson_n2n_pair_dataset(
    input_dir=kodak_clean_dir,
    output_dir_noisy1=kodak_noisy1_dir,
    output_dir_noisy2=kodak_noisy2_dir,
    lambda_value=10,
    seed1=42,
    seed2=123
);

Cartella input clean: /kaggle/input/datasets/sherylmehta/kodak-dataset
Output noisy1: /kaggle/working/n2n_poisson_datasets/kodak_lambda10/noisy1
Output noisy2: /kaggle/working/n2n_poisson_datasets/kodak_lambda10/noisy2
Lambda Poisson: 10
Immagini trovate: 24


100%|██████████| 24/24 [00:05<00:00,  4.57it/s]

Immagini processate: 24
PSNR medio clean/noisy1: 14.521698959155222
PSNR medio clean/noisy2: 14.52067115657807


In [8]:
create_poisson_n2n_pair_dataset(
    input_dir=bsd500_clean_dir,
    output_dir_noisy1=bsd500_noisy1_dir,
    output_dir_noisy2=bsd500_noisy2_dir,
    lambda_value=10,
    seed1=42,
    seed2=123
);

Cartella input clean: /kaggle/input/datasets/balraj98/berkeley-segmentation-dataset-500-bsds500/images
Output noisy1: /kaggle/working/n2n_poisson_datasets/bsd500_lambda10/noisy1
Output noisy2: /kaggle/working/n2n_poisson_datasets/bsd500_lambda10/noisy2
Lambda Poisson: 10
Immagini trovate: 500


100%|██████████| 500/500 [00:33<00:00, 14.89it/s]

Immagini processate: 500
PSNR medio clean/noisy1: 14.764408524351413
PSNR medio clean/noisy2: 14.76447551630501


In [9]:
create_poisson_n2n_pair_dataset(
    input_dir=div2k_train_clean_dir,
    output_dir_noisy1=div2k_noisy1_base_dir / "DIV2K_train_HR",
    output_dir_noisy2=div2k_noisy2_base_dir / "DIV2K_train_HR",
    lambda_value=10,
    seed1=42,
    seed2=123
)

Cartella input clean: /kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR
Output noisy1: /kaggle/working/n2n_poisson_datasets/div2k_lambda10/noisy1/DIV2K_train_HR
Output noisy2: /kaggle/working/n2n_poisson_datasets/div2k_lambda10/noisy2/DIV2K_train_HR
Lambda Poisson: 10
Immagini trovate: 800


100%|██████████| 800/800 [18:05<00:00,  1.36s/it]

Immagini processate: 800
PSNR medio clean/noisy1: 14.885058040577427
PSNR medio clean/noisy2: 14.885112648153047


([PosixPath('/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR/0001.png'),
  PosixPath('/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR/0002.png'),
  PosixPath('/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR/0003.png'),
  PosixPath('/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR/0004.png'),
  PosixPath('/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR/0005.png'),
  PosixPath('/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR/0006.png'),
  PosixPath('/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR/0007.png'),
  PosixPath('/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR/0008.png'),
  PosixPath('/kaggle/input/datas

In [10]:
create_poisson_n2n_pair_dataset(
    input_dir=div2k_valid_clean_dir,
    output_dir_noisy1=div2k_noisy1_base_dir / "DIV2K_valid_HR",
    output_dir_noisy2=div2k_noisy2_base_dir / "DIV2K_valid_HR",
    lambda_value=10,
    seed1=2024,
    seed2=3024
);

Cartella input clean: /kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_valid_HR/DIV2K_valid_HR
Output noisy1: /kaggle/working/n2n_poisson_datasets/div2k_lambda10/noisy1/DIV2K_valid_HR
Output noisy2: /kaggle/working/n2n_poisson_datasets/div2k_lambda10/noisy2/DIV2K_valid_HR
Lambda Poisson: 10
Immagini trovate: 100


100%|██████████| 100/100 [02:19<00:00,  1.39s/it]

Immagini processate: 100
PSNR medio clean/noisy1: 15.034311549188436
PSNR medio clean/noisy2: 15.033779524727118


In [11]:
# -----------------------------
# BSD500 noisy1/noisy2 split
# -----------------------------

bsd500_train_noisy1_dir = bsd500_noisy1_dir / "train"
bsd500_val_noisy1_dir   = bsd500_noisy1_dir / "val"
bsd500_test_noisy1_dir  = bsd500_noisy1_dir / "test"

bsd500_train_noisy2_dir = bsd500_noisy2_dir / "train"
bsd500_val_noisy2_dir   = bsd500_noisy2_dir / "val"
bsd500_test_noisy2_dir  = bsd500_noisy2_dir / "test"


# -----------------------------
# DIV2K noisy1/noisy2 split
# -----------------------------

div2k_train_noisy1_dir = div2k_noisy1_base_dir / "DIV2K_train_HR"
div2k_valid_noisy1_dir = div2k_noisy1_base_dir / "DIV2K_valid_HR"

div2k_train_noisy2_dir = div2k_noisy2_base_dir / "DIV2K_train_HR"
div2k_valid_noisy2_dir = div2k_noisy2_base_dir / "DIV2K_valid_HR"

In [12]:
# BSD500 completo
bsd500_train_noisy1_paths = get_image_paths(bsd500_train_noisy1_dir)
bsd500_val_noisy1_paths   = get_image_paths(bsd500_val_noisy1_dir)

# DIV2K subset per evitare problemi di RAM
div2k_train_noisy1_paths = get_image_paths(div2k_train_noisy1_dir)[:200]
div2k_val_noisy1_paths   = get_image_paths(div2k_valid_noisy1_dir)[:20]

train_noisy1_paths = bsd500_train_noisy1_paths + div2k_train_noisy1_paths
val_noisy1_paths   = bsd500_val_noisy1_paths + div2k_val_noisy1_paths

print("Training images:", len(train_noisy1_paths))
print("Validation images:", len(val_noisy1_paths))

Training images: 400
Validation images: 120


In [13]:
def get_corresponding_noisy2_path(noisy1_path, noisy1_root, noisy2_root):
    """
    Dato il path di una immagine noisy1, restituisce il path corrispondente
    della stessa immagine nella cartella noisy2.
    """

    noisy1_path = Path(noisy1_path)
    noisy1_root = Path(noisy1_root)
    noisy2_root = Path(noisy2_root)

    relative_path = noisy1_path.relative_to(noisy1_root)

    return noisy2_root / relative_path

In [14]:
def extract_paired_random_patches(
    noisy1_paths,
    noisy1_roots,
    noisy2_roots,
    patch_size=64,
    patches_per_image=10,
    seed=42
):
    """
    Estrae coppie di patch corrispondenti da noisy1 e noisy2.

    X = patch presa da noisy1
    Y = patch corrispondente presa da noisy2

    Queste coppie vengono usate per addestrare Noise2Noise:
        input  = X
        target = Y
    """

    rng = np.random.default_rng(seed)

    X_patches = []
    Y_patches = []

    for noisy1_path in tqdm(noisy1_paths):

        noisy1_path = Path(noisy1_path)

        matched_root = None
        matched_target_root = None

        # Capisce se l'immagine appartiene a BSD500 o DIV2K
        for root1, root2 in zip(noisy1_roots, noisy2_roots):
            try:
                noisy1_path.relative_to(root1)
                matched_root = Path(root1)
                matched_target_root = Path(root2)
                break
            except ValueError:
                continue

        if matched_root is None:
            print("Root non trovata per:", noisy1_path)
            continue

        noisy2_path = get_corresponding_noisy2_path(
            noisy1_path,
            matched_root,
            matched_target_root
        )

        if not noisy2_path.exists():
            print("Noisy2 non trovata:", noisy2_path)
            continue

        img1 = Image.open(noisy1_path).convert("RGB")
        img2 = Image.open(noisy2_path).convert("RGB")

        img1 = np.array(img1).astype(np.float32) / 255.0
        img2 = np.array(img2).astype(np.float32) / 255.0

        h, w, c = img1.shape

        if h < patch_size or w < patch_size:
            print("Immagine troppo piccola, saltata:", noisy1_path)
            continue

        for _ in range(patches_per_image):
            y = rng.integers(0, h - patch_size + 1)
            x = rng.integers(0, w - patch_size + 1)

            patch1 = img1[y:y + patch_size, x:x + patch_size, :]
            patch2 = img2[y:y + patch_size, x:x + patch_size, :]

            X_patches.append(patch1)
            Y_patches.append(patch2)

    X_patches = np.array(X_patches, dtype=np.float16)
    Y_patches = np.array(Y_patches, dtype=np.float16)

    print("X patches:", X_patches.shape)
    print("Y patches:", Y_patches.shape)

    return X_patches, Y_patches

In [15]:
X_train, Y_train = extract_paired_random_patches(
    noisy1_paths=train_noisy1_paths,
    noisy1_roots=[
        bsd500_train_noisy1_dir,
        div2k_train_noisy1_dir
    ],
    noisy2_roots=[
        bsd500_train_noisy2_dir,
        div2k_train_noisy2_dir
    ],
    patch_size=64,
    patches_per_image=5,
    seed=42
)

X_val, Y_val = extract_paired_random_patches(
    noisy1_paths=val_noisy1_paths,
    noisy1_roots=[
        bsd500_val_noisy1_dir,
        div2k_valid_noisy1_dir
    ],
    noisy2_roots=[
        bsd500_val_noisy2_dir,
        div2k_valid_noisy2_dir
    ],
    patch_size=64,
    patches_per_image=2,
    seed=123
)

100%|██████████| 400/400 [00:37<00:00, 10.75it/s]


X patches: (2000, 64, 64, 3)
Y patches: (2000, 64, 64, 3)


100%|██████████| 120/120 [00:02<00:00, 40.05it/s]

X patches: (240, 64, 64, 3)
Y patches: (240, 64, 64, 3)


In [16]:
X_train = X_train.astype(np.float32)
Y_train = Y_train.astype(np.float32)

X_val = X_val.astype(np.float32)
Y_val = Y_val.astype(np.float32)

In [17]:
import tensorflow as tf
from tensorflow.keras import layers, models

def conv_block(x, filters):
    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        kernel_initializer="he_normal"
    )(x)
    x = layers.LeakyReLU(negative_slope=0.1)(x)

    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        kernel_initializer="he_normal"
    )(x)
    x = layers.LeakyReLU(negative_slope=0.1)(x)

    return x


def build_n2n_unet(input_shape=(64, 64, 3), base_filters=32):
    inputs = layers.Input(shape=input_shape)

    c1 = conv_block(inputs, base_filters)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, base_filters * 2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, base_filters * 4)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    bottleneck = conv_block(p3, base_filters * 8)

    u3 = layers.UpSampling2D((2, 2))(bottleneck)
    u3 = layers.Concatenate()([u3, c3])
    c4 = conv_block(u3, base_filters * 4)

    u2 = layers.UpSampling2D((2, 2))(c4)
    u2 = layers.Concatenate()([u2, c2])
    c5 = conv_block(u2, base_filters * 2)

    u1 = layers.UpSampling2D((2, 2))(c5)
    u1 = layers.Concatenate()([u1, c1])
    c6 = conv_block(u1, base_filters)

    outputs = layers.Conv2D(
        3,
        kernel_size=1,
        padding="same",
        activation="sigmoid"
    )(c6)

    model = models.Model(inputs, outputs)

    return model

In [18]:
model = build_n2n_unet(
    input_shape=(64, 64, 3),
    base_filters=32
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        beta_1=0.9,
        beta_2=0.99,
        epsilon=1e-8
    ),
    loss="mse",
    metrics=["mae"]
)

model.summary()

2026-05-19 15:49:22.578101: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │      9,248 │ leaky_re_lu[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ leaky_re_lu_1[0]… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 32, 32,    │          0 │ conv2d_2[0][0]    │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │     36,928 │ leaky_re_lu_2[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_3       │ (None, 32, 32,    │          0 │ conv2d_3[0][0]    │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ leaky_re_lu_3[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_4       │ (None, 16, 16,    │          0 │ conv2d_4[0][0]    │
│ (LeakyReLU)         │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │    147,584 │ leaky_re_lu_4[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_5       │ (None, 16, 16,    │          0 │ conv2d_5[0][0]    │
│ (LeakyReLU)         │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 8, 8, 128) │          0 │ leaky_re_lu_5[0]… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 8, 8, 256) │    295,168 │ max_pooling2d_2[

 Total params: 1,946,947 (7.43 MB)

 Trainable params: 1,946,947 (7.43 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="/kaggle/working/n2n_poisson_lambda10_best.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    )
]

In [20]:
history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=30,
    batch_size=4,
    callbacks=callbacks
)

Epoch 1/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - loss: 0.0537 - mae: 0.1855
Epoch 1: val_loss improved from inf to 0.02496, saving model to /kaggle/working/n2n_poisson_lambda10_best.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 107s 208ms/step - loss: 0.0537 - mae: 0.1855 - val_loss: 0.0250 - val_mae: 0.1230 - learning_rate: 1.0000e-04
Epoch 2/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - loss: 0.0301 - mae: 0.1352
Epoch 2: val_loss improved from 0.02496 to 0.02420, saving model to /kaggle/working/n2n_poisson_lambda10_best.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 101s 202ms/step - loss: 0.0301 - mae: 0.1352 - val_loss: 0.0242 - val_mae: 0.1213 - learning_rate: 1.0000e-04
Epoch 3/30
500/500 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - loss: 0.0293 - mae: 0.1333
Epoch 3: val_loss improved from 0.02420 to 0.02344, saving model to /kaggle/working/n2n_poisson_lambda10_best.keras
500/500 ━━━━━━━━━━━━━━━━━━━━ 101s 202ms/step - loss: 0.0293 - mae: 0.1333 - val_loss: 0.0234 - val_mae: 0.1196 - learning_rate: 1

In [21]:
def pad_to_multiple_of_8(img):
    """
    Aggiunge padding riflesso a un'immagine affinché altezza e larghezza
    siano divisibili per 8.

    Questo è necessario perché la U-Net usa 3 livelli di MaxPooling,
    quindi le dimensioni devono essere compatibili con 2^3 = 8.
    """

    h, w, c = img.shape

    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8

    padded = np.pad(
        img,
        ((0, pad_h), (0, pad_w), (0, 0)),
        mode="reflect"
    )

    return padded, h, w


In [22]:
def denoise_dataset_with_n2n(
    model,
    noisy_dir,
    clean_dir,
    output_dir,
    csv_name="metrics.csv"
):
    """
    Applica il modello Noise2Noise a tutte le immagini rumorose di noisy_dir,
    salva le immagini denoised in output_dir e calcola PSNR/SSIM rispetto
    alle immagini clean corrispondenti.

    Il modello riceve in input una immagine noisy1 e produce una versione denoised.
    Le immagini clean vengono usate solo per la valutazione finale.
    """

    noisy_dir = Path(noisy_dir)
    clean_dir = Path(clean_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    noisy_paths = get_image_paths(noisy_dir)

    results = []

    for noisy_path in tqdm(noisy_paths):
        try:
            relative_path = noisy_path.relative_to(noisy_dir)
            clean_path = clean_dir / relative_path

            if not clean_path.exists():
                print("Clean non trovata:", clean_path)
                continue

            # Lettura immagine rumorosa
            noisy_img = Image.open(noisy_path).convert("RGB")
            noisy_np = np.array(noisy_img).astype(np.float32) / 255.0

            # Predizione N2N
            noisy_padded, original_h, original_w = pad_to_multiple_of_8(noisy_np)

            input_tensor = np.expand_dims(noisy_padded, axis=0)

            pred_padded = model.predict(input_tensor, verbose=0)[0]

            pred = pred_padded[:original_h, :original_w, :]
            pred = np.clip(pred, 0, 1)

            # Salvataggio immagine denoised
            save_path = output_dir / relative_path
            save_path = save_path.with_suffix(".png")
            save_path.parent.mkdir(parents=True, exist_ok=True)

            Image.fromarray((pred * 255).astype(np.uint8)).save(save_path)

            # Lettura clean
            clean_img = Image.open(clean_path).convert("RGB")
            clean_img = clean_img.resize((noisy_np.shape[1], noisy_np.shape[0]))
            clean_np = np.array(clean_img).astype(np.float32) / 255.0

            # Metriche noisy
            psnr_noisy = psnr(clean_np, noisy_np, data_range=1.0)
            ssim_noisy = ssim(clean_np, noisy_np, channel_axis=2, data_range=1.0)

            # Metriche N2N
            psnr_n2n = psnr(clean_np, pred, data_range=1.0)
            ssim_n2n = ssim(clean_np, pred, channel_axis=2, data_range=1.0)

            results.append({
                "filename": str(relative_path),
                "clean_path": str(clean_path),
                "noisy_path": str(noisy_path),
                "denoised_path": str(save_path),
                "psnr_clean_noisy": float(psnr_noisy),
                "ssim_clean_noisy": float(ssim_noisy),
                "psnr_clean_n2n": float(psnr_n2n),
                "ssim_clean_n2n": float(ssim_n2n),
                "psnr_gain": float(psnr_n2n - psnr_noisy),
                "ssim_gain": float(ssim_n2n - ssim_noisy),
            })

        except Exception as e:
            print("Errore su:", noisy_path)
            print(e)

    df = pd.DataFrame(results)

    csv_path = output_dir / csv_name
    df.to_csv(csv_path, index=False)

    print("CSV salvato in:", csv_path)

    if len(df) > 0:
        print("PSNR medio clean/noisy:", df["psnr_clean_noisy"].mean())
        print("PSNR medio clean/N2N:", df["psnr_clean_n2n"].mean())
        print("Gain PSNR medio:", df["psnr_gain"].mean())

        print("SSIM medio clean/noisy:", df["ssim_clean_noisy"].mean())
        print("SSIM medio clean/N2N:", df["ssim_clean_n2n"].mean())
        print("Gain SSIM medio:", df["ssim_gain"].mean())

    return df


In [23]:
model_flexible = build_n2n_unet(
    input_shape=(None, None, 3),
    base_filters=32
)

model_flexible.set_weights(model.get_weights())

In [24]:
df_kodak_n2n = denoise_dataset_with_n2n(
    model=model_flexible,
    noisy_dir=kodak_noisy1_dir,
    clean_dir=kodak_clean_dir,
    output_dir=n2n_output_dir / "kodak",
    csv_name="kodak_poisson_lambda10_n2n_metrics.csv"
)

100%|██████████| 24/24 [00:35<00:00,  1.50s/it]

CSV salvato in: /kaggle/working/n2n_poisson_outputs/kodak/kodak_poisson_lambda10_n2n_metrics.csv
PSNR medio clean/noisy: 14.521698844640133
PSNR medio clean/N2N: 25.535254460863058
Gain PSNR medio: 11.013555616222925
SSIM medio clean/noisy: 0.17992970254272223
SSIM medio clean/N2N: 0.7072696710626284
Gain SSIM medio: 0.5273399688303471


In [25]:
df_bsd500_n2n = denoise_dataset_with_n2n(
    model=model_flexible,
    noisy_dir=bsd500_test_noisy1_dir,
    clean_dir=bsd500_test_clean_dir,
    output_dir=n2n_output_dir / "bsd500_test",
    csv_name="bsd500_test_poisson_lambda10_n2n_metrics.csv"
)

100%|██████████| 200/200 [02:10<00:00,  1.54it/s]

CSV salvato in: /kaggle/working/n2n_poisson_outputs/bsd500_test/bsd500_test_poisson_lambda10_n2n_metrics.csv
PSNR medio clean/noisy: 17.626273001550697
PSNR medio clean/N2N: 25.01311127629582
Gain PSNR medio: 7.386838274745124
SSIM medio clean/noisy: 0.3185002916306257
SSIM medio clean/N2N: 0.7271783477067948
Gain SSIM medio: 0.40867805644869803


In [26]:
df_div2k_n2n = denoise_dataset_with_n2n(
    model=model_flexible,
    noisy_dir=div2k_valid_noisy1_dir,
    clean_dir=div2k_valid_clean_dir,
    output_dir=n2n_output_dir / "div2k_valid",
    csv_name="div2k_valid_poisson_lambda10_n2n_metrics.csv"
)

100%|██████████| 100/100 [16:45<00:00, 10.05s/it]

CSV salvato in: /kaggle/working/n2n_poisson_outputs/div2k_valid/div2k_valid_poisson_lambda10_n2n_metrics.csv
PSNR medio clean/noisy: 15.034311467049593
PSNR medio clean/N2N: 25.67902644534273
Gain PSNR medio: 10.644714978293134
SSIM medio clean/noisy: 0.2193335348367691
SSIM medio clean/N2N: 0.7602055323123932
Gain SSIM medio: 0.5408719953894615
